<a id="inizio"></a>
# Esercitazione 3 · Soluzioni

⏱ ~20 min &nbsp;·&nbsp; Dati: `../Dati/U.S. Electricity Prices.csv`, `../Dati/load_total_north_hourly_2024.xlsx`, `../Dati/TexasTurbine.csv`

**In questo notebook impariamo a**

- filtrare e raggruppare i prezzi dell'elettricità per stato
- portare il carico Terna da quartorario a giornaliero con `resample`
- disegnare un istogramma con Plotly Express

<a id="sez-1"></a>
## 1. Domande

Le cinque domande che seguono riguardano i notebook 07-09. A ciascuna si risponde in una riga, a parole
oppure provando il codice in una cella.

1. Su un DataFrame con indice `0, 1, 2, 3, ...`, quante righe restituiscono `df.loc[0:2]` e `df.iloc[0:2]`? Perché?
2. Cosa restituisce `df.groupby("Categoria")["Vendite"].sum()`? E `df.groupby("Categoria")` da solo?
3. Cosa fa `serie.resample("D").mean()` su una serie con un valore ogni 15 minuti? Che cosa serve perché funzioni?
4. A cosa serve `dayfirst=True` in `pd.to_datetime`? Fai l'esempio di `"01/02/2025"`.
5. Quando serve `how="left"` in `pd.merge`, invece del valore predefinito?

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Risposte**

1. `df.loc[0:2]` restituisce 3 righe, mentre `df.iloc[0:2]` ne restituisce 2. `loc` seleziona per etichetta e
   include l'etichetta finale dell'intervallo, invece `iloc` seleziona per posizione ed esclude la posizione
   finale, come lo slicing delle liste.
2. La prima espressione restituisce una Series che ha come indice le categorie e come valori la somma delle
   vendite di ciascuna. `df.groupby("Categoria")` da solo restituisce invece un oggetto GroupBy, che descrive i
   gruppi ma non calcola nulla finché non gli si applica una funzione di aggregazione come `sum()` o `mean()`.
3. Raggruppa i valori per giorno e ne calcola la media, quindi i 96 valori quartorari di ogni giornata diventano
   un valore solo. Perché funzioni, la serie deve avere un indice di date, che si ottiene per esempio con
   `set_index` su una colonna già convertita con `pd.to_datetime`.
4. Indica a pandas che nelle date ambigue il primo numero è il giorno, come si usa in Italia. Con
   `dayfirst=True` la stringa `"01/02/2025"` diventa il 1° febbraio 2025, mentre senza il parametro viene letta
   come 2 gennaio.
5. Serve quando vogliamo conservare tutte le righe della tabella di sinistra, comprese quelle che non hanno una
   corrispondenza nell'altra tabella; per queste righe le colonne che arrivano da destra valgono `NaN`. Con il
   valore predefinito `how="inner"`, invece, le righe senza corrispondenza vengono scartate.

</div>

<a id="sez-2"></a>
## 2. Esercizi

La cella qui sotto importa pandas e Plotly Express e legge i prezzi mensili dell'elettricità negli Stati
Uniti, che servono al primo esercizio. La colonna `price` è espressa in centesimi di dollaro per kWh, mentre
`stateDescription` contiene, oltre ai singoli stati, anche alcune regioni e il totale nazionale `U.S. Total`.

In [ ]:
import pandas as pd
import plotly.express as px

prezzi = pd.read_csv("../Dati/U.S. Electricity Prices.csv")
prezzi["date"] = pd.to_datetime(prezzi["date"])
prezzi.head()

<a id="es-e3-1"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.1 · Il prezzo residenziale per stato**

Vogliamo sapere in quali stati l'elettricità per le famiglie è costata di più nel 2023.

**Cosa fare.**

1. Metti in `residenziale_2023` le righe di `prezzi` che hanno `sectorName` uguale a `"residential"` e una
   data del 2023.
2. Calcola in `prezzo_medio` il prezzo medio per ogni valore di `stateDescription` e ordina il risultato
   dal prezzo più alto al più basso.

*Suggerimento:* le due condizioni vanno scritte ciascuna tra parentesi e unite con `&`, mentre l'anno di una colonna di date si legge con `.dt.year`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.1**

</div>

In [ ]:
residenziale = prezzi["sectorName"] == "residential"
anno_2023 = prezzi["date"].dt.year == 2023
residenziale_2023 = prezzi[residenziale & anno_2023]

prezzo_medio = residenziale_2023.groupby("stateDescription")["price"].mean().sort_values(ascending=False)
prezzo_medio.head()

In [ ]:
from corso import verifica
verifica("E3.1")

<a id="es-e3-2"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.2 · Il carico giornaliero di ottobre**

Il file Terna riporta il carico elettrico del Nord in MW, con un valore ogni 15 minuti e le righe
ordinate dalla più recente alla più vecchia. Il 27 ottobre quattro orari compaiono due volte, perché con il ritorno all'ora
solare l'ora dalle 2 alle 3 si ripete.

**Cosa fare.**

1. Ordina `carico` per `Date`, elimina le righe con un valore di `Date` ripetuto e imposta `Date` come indice.
2. Calcola in `giornaliero` la media giornaliera della colonna `Total Load [MW]` per il mese di ottobre 2024.
3. Metti in `giorno_max` il giorno con il carico medio più alto.

*Suggerimento:* per i duplicati si usa `drop_duplicates(subset="Date")`, mentre per la media si seleziona il mese con `.loc["2024-10", "Total Load [MW]"]` e si applica `resample("D")`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.2**

</div>

In [ ]:
carico = pd.read_excel("../Dati/load_total_north_hourly_2024.xlsx")
carico = carico.sort_values("Date").drop_duplicates(subset="Date").set_index("Date")

giornaliero = carico.loc["2024-10", "Total Load [MW]"].resample("D").mean()
giorno_max = giornaliero.idxmax()
giorno_max

In [ ]:
from corso import verifica
verifica("E3.2")

<a id="es-e3-3"></a>
<div style="background:rgba(92,184,92,0.12); border-left:4px solid #5cb85c; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✏️ Esercizio E3.3 · L'istogramma della potenza della turbina**

Il file `TexasTurbine.csv` contiene la potenza prodotta da una turbina eolica in Texas, con un valore per
ogni ora dell'anno. Un istogramma mostra come si distribuiscono questi valori, comprese le ore in cui la
turbina è ferma.

**Cosa fare.**

1. Disegna in `fig` l'istogramma della colonna `System power generated | (kW)` con `px.histogram`.
2. Metti in `ore_ferme` il numero di ore in cui la potenza è 0.

*Suggerimento:* il confronto `== 0` sulla colonna produce una maschera booleana, e `.sum()` applicato alla maschera conta i valori `True`.

</div>

<div style="background:rgba(46,159,214,0.12); border-left:4px solid #2e9fd6; padding:12px 14px; border-radius:4px; margin:8px 0 12px 0;">

**✅ Soluzione E3.3**

</div>

In [ ]:
turbina = pd.read_csv("../Dati/TexasTurbine.csv")

fig = px.histogram(turbina, x="System power generated | (kW)", nbins=30,
                   title="Potenza oraria della turbina")
fig.show()

ore_ferme = (turbina["System power generated | (kW)"] == 0).sum()
ore_ferme

In [ ]:
from corso import verifica
verifica("E3.3")

---

**Fine dell'esercitazione 3.** Precedente: [09 · Plotly per serie storiche](09_Plotly.ipynb) &nbsp;·&nbsp; Prossimo: [10 · Agenti per il coding](10_Agenti_per_il_coding.ipynb)

<sub>Blocco 3 · Analisi dei dati con pandas e visualizzazione · Giornata 2 · Aula Avanzata</sub>